# mama — Google Colab Export

This notebook was generated by mama. It reproduces your training run in Google Colab.

Run all cells (Runtime → Run all) to re-run training with your saved configuration.

In [ ]:
# --- TRAINING CONFIG (injected by mama) ---
# If this cell is empty, no training_config.json was found.
# Fill in the fields below manually, or upload a training_config.json
# and update the CONFIG_JSON string.

CONFIG_JSON = '''{}'''

## 1. Install Dependencies

In [ ]:
import subprocess, sys, json, os
from pathlib import Path

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

run("pip install -q torch transformers datasets accelerate tensorboard xformers")

## 2. Load Training Config

In [ ]:
if CONFIG_JSON.strip():
    cfg = json.loads(CONFIG_JSON)
    print("Loaded config from embedded data:")
else:
    # No config was embedded — define your settings here
    print("No embedded config found. Using default placeholder settings.")
    print("Edit the CONFIG dictionary below with your training parameters.")
    cfg = {
        "model_name_or_path": "bert-base-uncased",
        "dataset_path": "/content/dataset",
        "output_dir": "/content/outputs",
        "text_column": "text",
        "learning_rate": 2e-4,
        "per_device_train_batch_size": 2,
        "num_train_epochs": 3,
        "max_seq_length": 2048,
    }

print(json.dumps(cfg, indent=2))

model_name = cfg.get("model_name_or_path", "")
dataset_path = cfg.get("dataset_path", "")
output_dir = Path(cfg.get("output_dir", "/content/outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

if not model_name:
    raise ValueError("model_name_or_path is required. Set it in the config above.")
if not dataset_path:
    raise ValueError("dataset_path is required. Set it in the config above.")

## 3. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_qlora = cfg.get("use_qlora", False)
use_lora = cfg.get("use_lora", False)

load_kwargs = {"torch_dtype": "auto"}
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)

## 4. Load & Tokenize Dataset

In [ ]:
import datasets

print(f"Loading dataset: {dataset_path}")

path_obj = Path(dataset_path)
if path_obj.exists():
    if path_obj.is_dir():
        dataset = datasets.load_from_disk(str(path_obj))
    else:
        dataset = datasets.load_dataset(
            "json" if path_obj.suffix == ".jsonl" else "csv",
            data_files=str(path_obj),
            split="train",
        )
else:
    dataset = datasets.load_dataset(dataset_path, split="train")

# --- Auto-detect text column ---
text_column = cfg.get("text_column", "")
available_columns = dataset.column_names
print(f"Available columns: {available_columns}")

if text_column and text_column in available_columns:
    print(f"Using configured text_column: '{text_column}'")
else:
    # Try common text column names in priority order
    for candidate in ["text", "prompt", "instruction", "input", "content", "messages", "conversation", "sentence"]:
        if candidate in available_columns:
            text_column = candidate
            break
    if not text_column or text_column not in available_columns:
        # Fall back to the first string-type column
        for col in available_columns:
            if dataset.features[col].dtype == "string":
                text_column = col
                break
    print(f"Auto-selected text_column: '{text_column}'")
    if not text_column:
        raise ValueError(f"Could not determine text column. Available columns: {available_columns}. "
                         "Set 'text_column' in the config cell above.")

max_samples = cfg.get("max_samples", None)
if max_samples:
    dataset = dataset.select(range(min(max_samples, len(dataset))))

column_names = set(dataset.column_names)
is_chat = "messages" in column_names or "conversations" in column_names
is_prompt_completion = {"prompt", "completion"} <= column_names

def format_text(examples):
    if is_chat:
        # Conversation format — apply chat template
        key = "messages" if "messages" in column_names else "conversations"
        texts = [tokenizer.apply_chat_template(msg, tokenize=False) for msg in examples[key]]
    elif is_prompt_completion:
        texts = [p + " " + c for p, c in zip(examples["prompt"], examples["completion"])]
    else:
        texts = examples[text_column]
    return {"text": texts}

dataset = dataset.map(format_text, batched=True, remove_columns=list(column_names))

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=cfg.get("max_seq_length", 2048),
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(f"Dataset ready: {len(tokenized_dataset)} samples")

## 5. Configure & Run Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=cfg.get("per_device_train_batch_size", 2),
    gradient_accumulation_steps=cfg.get("gradient_accumulation_steps", 4),
    learning_rate=cfg.get("learning_rate", 2e-4),
    num_train_epochs=cfg.get("num_train_epochs", 3),
    max_steps=cfg.get("max_steps", -1),
    warmup_steps=cfg.get("warmup_steps", 100),
    logging_steps=cfg.get("logging_steps", 10),
    save_steps=cfg.get("save_steps", 500),
    lr_scheduler_type=cfg.get("lr_scheduler_type", "cosine"),
    gradient_checkpointing=cfg.get("gradient_checkpointing", False),
    bf16=cfg.get("bf16", False),
    logging_dir=str(output_dir / "logs"),
    report_to="tensorboard",
    ddp_find_unused_parameters=False if cfg.get("use_lora", False) else None,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

## 6. Save Model

In [ ]:
final_dir = output_dir / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Model saved to {final_dir}")
print("\nDone! Download your outputs from the Files panel on the left.")